# Multi-agentic System using Agent Harness

_Building multi-agentic system using an open source and popular agent harness._

Build an agent to perform deep research on a given topic leveraging the built-in capabilities (such as core tool calling loop, execution environment, context management, task delegation and steering) provided by an open source and popular agent harness.

**Prerequisites:**

- Refer to README installation section to install package `deepagents`.

- Register for Ollama API key at https://ollama.com/ for agent to perform web search. 
    - Create a free Ollama account by signing-in using one of the supported options including Google credential. 
    - Go to `Settings --> Keys`, click on `Add API Keys` button, enter an appropriate name such as `web_search` in the field `API Key Name` and click on `Add API key` button. As the generated value can be viewed only once, copy the key to the clipboard and store it somewhere safe.
    - Create a file named `.env` in the root directory of the workspace. Enter the below mentioned content in the file and replace the placeholder with the created key (remove the angle bracket, too) and save the file.

        ```
        # .env
        OLLAMA_API_KEY=<API KEY>
        ```

In [ ]:
# Imports packages

import os
import pathlib
from dotenv import load_dotenv
from phoenix.otel import register

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware

from langgraph.checkpoint.memory import MemorySaver
from deepagents.backends import FilesystemBackend

In [ ]:
base_dir = pathlib.Path.cwd()   # Gets the current working directory (to be used later)
base_dir                        # prints the absolute path for reference

## TRACING
Enables instrumentation for effective observability.

In [ ]:
# Sets the environment variables for Phoenix to read from

os.environ["PHOENIX_API_KEY"] = "dummy-api-key"     # No valid API key is required for self-hosted service
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:4317"  # The collector endpoint to which spans will be exported.

In [4]:
# Creates an OpenTelemetry TracerProvider for enabling OpenInference tracing.
tracer_provider = register(
    project_name="deep-agent",  # Any appropriate Phoenix project name for observability
    auto_instrument=True        # Performs automatic instrumentations
    )  

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: deep-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## Tools
Defines custom tools the agent can call in addition to harness provided built-in tools (such as file management and subagent spawning tools).

In [ ]:
# Loads all the variables as environment variables. (Also loads Ollama API keys.)
load_dotenv()

from ollama import web_search, web_fetch    # These helper functions sets the API keys while tool calling

internet_search = {'web_search': web_search, 'web_fetch': web_fetch}


## Model

In [ ]:
os.environ["OPENAI_API_KEY"] = "dummy_api_key"

# NOTE: All the agent harness capabilities work best on large language models.
# But in this experimental setup, a small language model is used for demonstation purpose
OLLAMA_MODEL = "qwen3.5:2b" 

OLLAMA_OPENAI_ENDPOINT = "http://localhost:11434/v1"    # OpenAI compatible model endpoint

In [ ]:
# Initializes model (using OpenAI interface)
model = ChatOpenAI(model=OLLAMA_MODEL, base_url=OLLAMA_OPENAI_ENDPOINT)

In [ ]:
# Compact system prompt to reduce token load while preserving behavior.
research_instructions = """You are a LangGraph research assistant. Answer directly and concisely.
Use langgraph-docs first; use web_search/web_fetch only if docs are insufficient or recency is required.
Limit web_fetch to at most 2 pages and cite sources briefly when used."""

## Middleware
Extra middleware merge into the Deep Agents stack; an instance whose `.name` matches a built-in entry replaces it in place, anything else lands after the last core middleware entry and before the profile, prompt-caching, and memory.


In [9]:
# Adds a structured task planning tool `write_todos` by opting-in the `TodoListMiddleware`.
todolist_middleware = TodoListMiddleware()

## Subagents
Delegates isolated tasks and avoids context bloat in the main agent. In this experiment, main agent automatically adds a synchronous general-purpose subagent. Refer `Event Streaming` section.

## Skills
Provides deep agent with new capabilities and expertise on-demand by containing detailed instructions, best practices, scripts, reference documents and templates on how to complete tasks for the agent to determine which skill is useful for the current prompt. The progressive disclosure is achieved by avoiding context bloat by loading only summaries of skills at startup and reading full instructions when a task requires them. Skills are sharable across agents and projects, and multiple skills can be composed in a single agent so each one covers a distinct capability.

Each skill resides in its respective subdirectory inside a top-level skills directory relative to the backend root. Each skill subdirectory contains a `SKILL.md` file: a markdown file with YAML frontmatter (name and description) followed by instructions the agent follows when the skill is activated.

In this experiment, knowledge related to LangGraph is stored as document in the respective directory.

In [10]:
# Sets top-level skills directory; 
# Paths must be specified using forward slashes and are relative to the backend’s root.
skills = ["/skills/"]   

## Memory
Provide deep agent an extra context. `AGENTS.md` file gets loaded at startup from the configured backend. It is an updatable memory and the updates get stored back in the same configured backend.

In [11]:
memory=["/AGENTS.md"]

## Backend
Exposes the deep agent a filesystem surface via tools (like `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, and `grep`) that operate through a pluggable in-built and custom backends.

Popular built-in backends:
- StateBackend: A thread-scoped (default) filesystem backend stored in langgraph state.
- StoreBackend: A filesystem that provides long-term storage that is persisted across threads.
- FilesystemBackend: The local machine’s filesystem.

In [12]:
backend = FilesystemBackend(root_dir=base_dir)

## Event Steaming
Exposes real-time agent-runs as typed projections for messages, tool calls, values, and output for both coordinator (via `stream.messages`, `stream.values`, tool calls, custom updates) and subagnets (via `stream.subagents`).

In [14]:
agent = create_deep_agent(
    model=model,                # "ollama:qwen3.5:4b",   # "ollama:north-mini-code-1.0",
    tools=[web_search, web_fetch],
    system_prompt= research_instructions,
    middleware=[todolist_middleware],  # Gives the agent `write_todos` tool for maintaining a structured task list during execution.
    # subagents=[], # Deep Agents automatically adds a synchronous general-purpose subagent having filesystem tools by default
    skills=skills,       # Directory containing skills each lives in respective subdirectory OR str(Path(base_dir) / "skills/")
    memory=memory,     
    checkpointer=MemorySaver(),      # Required
    backend=backend,       
)

In [ ]:
# NOTE THAT THE FOLLOWING STEP MAY TAKE AROUND 15-20 MINUTES ON CPU

user_query= "Explain what LangGraph is, its architecture and desiign patterns. Also, research on recent developments around LangChain ecosystem for building multi-agent systems and show your findings with references."
thread_id = 1

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": user_query}]},
    config={"configurable": {"thread_id": thread_id}},
    version="v3",
)

coordinator_messages: list[str] = []
coordinator_tool_names: list[str] = []

for message in stream.messages:
    print("[coordinator]", message.text)
    coordinator_messages.append(message.text)

for subagent in stream.subagents:
    for message in subagent.messages:
        print(f"[subagent: {subagent.name}]", message.text)

    for call in subagent.tool_calls:
        print(f"[subagent {subagent.name} tool]", call.tool_name, call.input)
        for delta in call.output_deltas:
            print(delta, end="", flush=True)

        if call.completed and call.error is None:
            print(call.output)
        elif call.error is not None:
            print(call.error)

for call in stream.tool_calls:
    print("[coordinator tool]", call.tool_name, call.input)
    print(call.completed, call.error)
    coordinator_tool_names.append(call.tool_name)


[coordinator] I'll help you understand LangGraph, its architecture, and recent developments for multi-agent systems. Let me start by gathering information from the available skills and then conduct focused research.

## LangGraph Overview

**What is LangGraph?**

LangGraph is a library built on LangChain for building structured, scalable, stateful multi-agent workflows. It extends LangChain's simple linear chains to support directed acyclic graphs (DAGs) of nodes, enabling complex workflows with loops, conditions, and branching.

---

## LangGraph Architecture & Core Design Patterns

Let me retrieve this information from the langgraph-docs skill:

---

### 📋 **Architecture Components**
LangGraph's architecture consists of:

```
LangGraph Core Components
├── Nodes (functions or Python classes)
├── Edges (transitions between nodes)
├── State (shared data structure - pydantic models)
├── Graph (directed graph structure)
├── Compiled Graph (optimized runtime)
└── Human-in-the-Loop nodes (s